In [59]:
import yfinance as yf

In [61]:
data = yf.Ticker('SPY')

In [63]:
spy = data.history(period = '15y', auto_adjust= False)

In [64]:
spy

,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,Capital Gains
Date,,,,,,,,,
2010-12-27 00:00:00-05:00,125.129997,125.769997,125.040001,125.650002,96.121880,58126000,0.000,0.0,0.0
2010-12-28 00:00:00-05:00,125.900002,125.949997,125.500000,125.830002,96.259583,55309100,0.000,0.0,0.0
2010-12-29 00:00:00-05:00,125.980003,126.199997,125.900002,125.919998,96.328438,58033100,0.000,0.0,0.0
2010-12-30 00:00:00-05:00,125.800003,126.129997,125.529999,125.720001,96.175468,76616900,0.000,0.0,0.0
2010-12-31 00:00:00-05:00,125.529999,125.870003,125.330002,125.750000,96.198364,91218900,0.000,0.0,0.0
...,...,...,...,...,...,...,...,...,...
2025-12-19 00:00:00-05:00,676.590027,681.090027,676.469971,680.590027,680.590027,103599500,1.993,0.0,0.0
2025-12-22 00:00:00-05:00,683.940002,685.359985,680.590027,684.830017,684.830017,69556700,0.000,0.0,0.0
2025-12-23 00:00:00-05:00,683.919983,688.200012,683.869995,687.960022,687.960022,64840000,0.000,0.0,0.0


In [65]:
spy["sm21"] = spy["Adj Close"].rolling(21).mean()
spy["sm50"] = spy["Adj Close"].rolling(50).mean()

In [69]:
spy["distancia sm21"] = (spy["sm21"] - spy["Adj Close"])/spy["sm21"]
spy["distancia sm50"] = (spy["sm50"] - spy["Adj Close"])/spy["sm50"]

In [71]:
spy = spy.drop(["sm21", "sm50", "High", "Low", "Close", "Open", "Dividends", "Stock Splits", "Capital Gains"], axis = 1)

In [73]:
spy["target"] = spy["Adj Close"].pct_change().shift(-1)

In [75]:
spy = spy.dropna()

In [77]:
spy

,Adj Close,Volume,distancia sm21,distancia sm50,target
Date,,,,,
2011-03-08 00:00:00-05:00,101.423325,174615000,-0.000546,-0.019831,-0.001433
2011-03-09 00:00:00-05:00,101.277969,153806000,0.001038,-0.017314,-0.018506
2011-03-10 00:00:00-05:00,99.403694,301291800,0.018598,0.002143,0.006926
2011-03-11 00:00:00-05:00,100.092178,225621800,0.011292,-0.004010,-0.006037
2011-03-14 00:00:00-04:00,99.487885,234974100,0.016458,0.002715,-0.011457
...,...,...,...,...,...
2025-12-18 00:00:00-05:00,674.476929,108650100,0.001655,-0.002325,0.009063
2025-12-19 00:00:00-05:00,680.590027,103599500,-0.005981,-0.011067,0.006230
2025-12-22 00:00:00-05:00,684.830017,69556700,-0.009816,-0.016347,0.004570


In [79]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

In [81]:
from sklearn.model_selection import train_test_split

In [101]:
from sklearn.ensemble import RandomForestRegressor

In [85]:
x = spy.drop(["target"],axis=1) 
y = spy["target"]

In [87]:
standar = StandardScaler()

In [89]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2, random_state = 42,shuffle = False) 

In [91]:
x_train_scaled = standar.fit_transform(x_train)
x_test_scaled = standar.transform(x_test)

In [107]:
model = RandomForestRegressor(n_estimators = 150, criterion="absolute_error",random_state = 42)

In [109]:
model.fit(x_train_scaled, y_train)

RandomForestRegressor(criterion='absolute_error', n_estimators=150,
                      random_state=42)

In [113]:
predict = model.predict(x_test_scaled)

In [117]:
from sklearn.model_selection import cross_val_score

In [119]:
scores = cross_val_score(model, x_train_scaled, y_train,
                         scoring="neg_root_mean_squared_error",
                         cv=10)
scores

array([-0.01465532, -0.00852069, -0.00666033, -0.00918683, -0.00830189,
       -0.00771286, -0.00932116, -0.02121013, -0.01075463, -0.01526319])

In [120]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [148]:
import numpy as np

In [124]:
mse_model = mean_squared_error(y_test, predict)
mae_model = mean_absolute_error(y_test, predict)

In [162]:
print(f'mse = {np.sqrt(mse_model)}')
print(f'mae = {mae_model}')

mse = 0.011360221034943281
mae = 0.00873129363988827


In [142]:
print(f"Mean RMSE: {scores.mean():.4f}")

Mean RMSE: -0.0112


In [ ]:
#ahora que no hay sobreajuste, vamos a medir si hay mejores parametros

In [164]:
pipe = Pipeline([
    ("normalizacion", StandardScaler()),
    ("modelo", RandomForestRegressor(random_state=42))
])

In [176]:
param_grid = {
    'modelo__max_depth': [5, 10, 20],        
    'modelo__min_samples_leaf': [1, 15, 30], 
    'modelo__max_features': [0.33, 1.0],     
    'modelo__n_estimators': [100, 150, 200]  
}

In [178]:
from sklearn.model_selection import GridSearchCV

In [180]:
grid_search = GridSearchCV(pipe, param_grid, cv=3)

In [182]:
grid_search.fit(x_train,y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('normalizacion', StandardScaler()),
                                       ('modelo',
                                        RandomForestRegressor(random_state=42))]),
             param_grid={'modelo__max_depth': [5, 10, 20],
                         'modelo__max_features': [0.33, 1.0],
                         'modelo__min_samples_leaf': [1, 15, 30],
                         'modelo__n_estimators': [100, 150, 200]})

In [226]:
test = grid_search.predict(x_test)

In [228]:
acert_mae = mean_absolute_error(test, y_test)

In [230]:
acert_mae

0.006698307506177152

In [232]:
mejor_modelo = grid_search.best_estimator_

scores_rmse = cross_val_score(mejor_modelo, x_train, y_train,
                              scoring="neg_root_mean_squared_error",
                              cv=10)

rmse_promedio = -scores_rmse.mean()
print(f"RMSE Promedio (CV): {rmse_promedio:.4f}")

RMSE Promedio (CV): 0.0105


In [234]:
acert_mse = mean_squared_error(test, y_test)
np.sqrt(acert_mse)

0.009726193068752276

In [ ]:
#son muy parecidos, no hay sobreajuste.